In [ ]:
# Run this cell first in Google Colab after choosing a GPU runtime.
import subprocess, sys, os
from pathlib import Path
if not Path('/content').is_dir():
    raise RuntimeError('This launcher is for Colab. On your Mac, use financial_sentiment_bert.ipynb instead.')
project = Path('/content/financial-sentiment')
if not project.exists():
    subprocess.run(['git','clone','https://github.com/Cameron-Batts/Financial-Sentiment-BERT-Finetuning.git',str(project)], check=True)
os.chdir(project)
# Preserve Colab's installed GPU-capable PyTorch build; record its version at export.
subprocess.run([sys.executable,'-m','pip','install','transformers==4.57.1','accelerate==1.11.0'], check=True)
from google.colab import files
os.chdir('/content')
uploaded = files.upload()
candidates = [name for name in uploaded if name in ('all-data.csv','all-data(1).csv')]
if len(candidates) != 1:
    raise ValueError('Upload exactly one original all-data.csv or all-data(1).csv file.')
(project/'data/all-data.csv').write_bytes(uploaded[candidates[0]])
del uploaded
os.chdir(project)
print('Project and private runtime data are ready. Continue with the cells below.')


# Financial Sentiment Classification with BERT Fine-Tuning
**Cameron Batts**

Python · NLP · Transformers · PyTorch · BERT · FinBERT · scikit-learn

## Overview
Compare bag-of-words classifiers, an off-the-shelf sentiment checkpoint, head-only BERT adaptation and full BERT fine-tuning on financial-news sentiment. Inspect an external FinBERT model separately because its training data may overlap the evaluation corpus.

## Business context
Classifying financial text can help prioritize news for review. Model selection should account for minority-class recall, domain adaptation, compute cost and evaluation independence. This experiment does not measure investment returns or deployed business impact.

## Data and attribution
The supplied Financial PhraseBank-style CSV contains 4,846 labeled sentences. Source: Malo et al. (2014), *Good debt or bad debt: Detecting semantic orientations in economic texts*, https://doi.org/10.1002/asi.23062 . A 50% random sample yields 2,423 sentences. Raw text and saved person-name examples are omitted from this package. Place a permitted copy of `all-data.csv` under `data/` before running.

## MPS rerun evidence — October 22, 2026
The user-supplied MPS export records ten completed epochs for head-only adaptation and full fine-tuning on the original 485-example test split. Fine-tuned BERT reached 0.8412 accuracy, 0.8199 macro F1 and 0.8439 weighted F1, compared with historical rounded values 0.87, 0.84 and 0.87. The rerun improves accuracy by 8.04 percentage points over logistic regression. Validation loss selected epoch 1 for the full model (0.3956); later validation losses increased as training loss fell, indicating overfitting. Head-only adaptation selected epoch 10. Aggregate arithmetic, dataset hash, checkpoint revisions, baseline reports and logged checkpoint selection were checked. Raw predictions, model weights and the executed notebook were not supplied, so the training was not independently replayed here. Two unique test sentences also occur in training, and external FinBERT may have seen the evaluation corpus.

Historical rounded comparisons remain in reports/model_comparison.csv. Supplied rerun metrics are stored separately in reports/gpu_rerun_metrics.json.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn import naive_bayes
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
news_df = pd.read_csv(root / 'data' / 'all-data.csv', names=['sentiment', 'content'], encoding='latin-1')
news_df = news_df.sample(frac=0.5, random_state=41)
print(news_df['sentiment'].value_counts())


## Train, validation and test split
Use the original random_state=41 split: 1,453 training, 485 validation and 485 test sentences. The split is not stratified and does not deduplicate related sentences. Test support is 51 negative, 297 neutral and 137 positive. A neutral-only prediction would therefore have about 61.2% accuracy, making class-level and macro metrics essential.

In [ ]:

X = news_df["content"]
y = news_df["sentiment"]

# First split: 60% train, 40% temp
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=41)

# Second split: cut the 40% temp in half, so 20% validation and 20% test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=41)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))


## Bag-of-words benchmarks
Fit unigram/bigram vocabulary only on training data, with English stopword removal. Evaluate Naive Bayes and logistic regression on the same test set.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words='english')
X_train_vectors = vectorizer.fit_transform(X_train)
X_test_vectors = vectorizer.transform(X_test)

In [ ]:
model_nb = naive_bayes.MultinomialNB()
model_nb.fit(X_train_vectors, y_train)

In [ ]:

y_pred_nb = model_nb.predict(X_test_vectors)
print(classification_report(y_test, y_pred_nb, zero_division=0.0))

User-supplied MPS export; formatted from aggregate metrics, not executed in this session.
              precision    recall  f1-score   support

    negative       0.57      0.49      0.53        51
     neutral       0.80      0.88      0.84       297
    positive       0.67      0.55      0.61       137

Accuracy: 0.7464
Macro F1: 0.6565
Weighted F1: 0.7385


In [ ]:
model_lr = LogisticRegression()
model_lr.fit(X_train_vectors, y_train)

In [ ]:
y_pred_lr = model_lr.predict(X_test_vectors)
print(classification_report(y_test, y_pred_lr, zero_division=0.0))

User-supplied MPS export; formatted from aggregate metrics, not executed in this session.
              precision    recall  f1-score   support

    negative       0.65      0.39      0.49        51
     neutral       0.76      0.97      0.85       297
    positive       0.81      0.45      0.58       137

Accuracy: 0.7608
Macro F1: 0.6396
Weighted F1: 0.7365


## Off-the-shelf sentiment checkpoint
Load the checkpoint identifier used in the original analysis. It is a customer-comment sentiment model rather than a financial-domain model. Inference requires downloading model weights. Keep evaluation text local except for model downloads.

In [ ]:
import torch
from transformers import pipeline, AutoConfig, set_seed
set_seed(42)
if torch.backends.mps.is_available():
    device = 'mps'
elif torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'
bert_path = 'MarieAngeA13/Sentiment-Analysis-BERT'
bert_revision = '938fb4267e118e6f892d9825253fd28893d95966'
finbert_revision = '0bc2ac10dda6929bcb1f77ccc43c7ec5c424f031'
REQUIRE_ACCELERATOR = globals().get('REQUIRE_ACCELERATOR', True)
if REQUIRE_ACCELERATOR and device == 'cpu':
    raise RuntimeError('Select a GPU runtime or use a Mac with MPS before BERT execution. Baselines can run on CPU.')
checkpoint_config = AutoConfig.from_pretrained(bert_path, revision=bert_revision)
assert {int(k): v.lower() for k, v in checkpoint_config.id2label.items()} == {0:'negative', 1:'neutral', 2:'positive'}
print('Device:', device, 'Checkpoint revision:', bert_revision)
bert = pipeline(model=bert_path, device=device, revision=bert_revision)


In [ ]:
y_pred_bert = [pred["label"] for pred in bert(X_test.to_list())]
print(classification_report(y_test, y_pred_bert, zero_division=0.0))

User-supplied MPS export; formatted from aggregate metrics, not executed in this session.
              precision    recall  f1-score   support

    negative       0.53      0.39      0.45        51
     neutral       0.68      0.90      0.77       297
    positive       0.57      0.21      0.31       137

Accuracy: 0.6536
Macro F1: 0.5105
Weighted F1: 0.6080


## Prepare supervised BERT adaptation
Map negative, neutral and positive to 0, 1 and 2; tokenize each split and wrap tensors in a PyTorch dataset. Before reproducing, inspect the checkpoint label mapping: overriding label metadata does not itself reorder pretrained classifier weights. Verify the original checkpoint head semantics against this explicit mapping, especially for the frozen-base comparison.

In [ ]:
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

bert_path = "MarieAngeA13/Sentiment-Analysis-BERT"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_path, revision=bert_revision)


In [ ]:
sentiment2int = {"negative": 0, "neutral": 1, "positive": 2}
int2sentiment = {value: key for key, value in sentiment2int.items()}

In [ ]:
def tokenize_and_tensorize(text, labels, tokenizer, label_mapping):
    encodings = tokenizer(
        text.tolist(),
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    label_tensor = torch.tensor(labels.map(label_mapping).tolist(), dtype=torch.long)
    return encodings, label_tensor


train_encodings, y_train_tensor = tokenize_and_tensorize(
    X_train, y_train, bert_tokenizer, sentiment2int
)
val_encodings, y_val_tensor = tokenize_and_tensorize(
    X_val, y_val, bert_tokenizer, sentiment2int
)
test_encodings, y_test_tensor = tokenize_and_tensorize(
    X_test, y_test, bert_tokenizer, sentiment2int
)


In [ ]:
class FinancialNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, index):
        item = {name: values[index] for name, values in self.encodings.items()}
        item["labels"] = self.labels[index]
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = FinancialNewsDataset(train_encodings, y_train_tensor)
val_dataset = FinancialNewsDataset(val_encodings, y_val_tensor)
test_dataset = FinancialNewsDataset(test_encodings, y_test_tensor)


## Head-only adaptation
Freeze the BERT base and train its classification head for ten epochs, with batch size 32. Evaluate and save each epoch. With no custom selection metric provided, best-checkpoint loading uses validation loss rather than test scores or macro F1. This selects among saved checkpoints; it does not guarantee absence of overfitting.

In [ ]:
model_frozen = AutoModelForSequenceClassification.from_pretrained(
    bert_path,
    revision=bert_revision,
    num_labels=3,
    id2label=int2sentiment,
    label2id=sentiment2int,
    ignore_mismatched_sizes=True,
    attn_implementation="eager"
)

# Freeze the BERT base model so only the classification head trains
for param in model_frozen.bert.parameters():
    param.requires_grad = False

training_args_frozen = TrainingArguments(
    output_dir="./results_frozen",
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_strategy="epoch",
    report_to="none",
    seed=42,
    data_seed=42,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    dataloader_pin_memory=(device == "cuda")
)

trainer_frozen = Trainer(
    model=model_frozen,
    args=training_args_frozen,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
trainer_frozen.train()

pred_output_frozen = trainer_frozen.predict(test_dataset)
y_pred_frozen = np.argmax(pred_output_frozen.predictions, axis=1)

print(classification_report(
    y_test_tensor.numpy(),
    y_pred_frozen,
    labels=list(int2sentiment.keys()),
    target_names=list(int2sentiment.values()),
    zero_division=0.0
))

User-supplied MPS export; formatted from aggregate metrics, not executed in this session.
              precision    recall  f1-score   support

    negative       0.52      0.59      0.55        51
     neutral       0.75      0.82      0.78       297
    positive       0.59      0.43      0.50       137

Accuracy: 0.6866
Macro F1: 0.6101
Weighted F1: 0.6774


## Full model fine-tuning
Initialize a separate copy and leave all parameters trainable. Use the same epoch count, batch size, validation evaluation and checkpoint selection procedure. GPU acceleration is recommended; checkpoint saving can use substantial disk space.

In [ ]:

model_unfrozen = AutoModelForSequenceClassification.from_pretrained(
    bert_path,
    revision=bert_revision,
    num_labels=3,
    id2label=int2sentiment,
    label2id=sentiment2int,
    ignore_mismatched_sizes=True,
    attn_implementation="eager"
)

training_args_unfrozen = TrainingArguments(
    output_dir="./results_unfrozen",
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_strategy="epoch",
    report_to="none",
    seed=42,
    data_seed=42,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    dataloader_pin_memory=(device == "cuda")
)

trainer_unfrozen = Trainer(
    model=model_unfrozen,
    args=training_args_unfrozen,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:

trainer_unfrozen.train()

pred_output_unfrozen = trainer_unfrozen.predict(test_dataset)
y_pred_unfrozen = np.argmax(pred_output_unfrozen.predictions, axis=1)

print(classification_report(
    y_test_tensor.numpy(),
    y_pred_unfrozen,
    labels=list(int2sentiment.keys()),
    target_names=list(int2sentiment.values()),
    zero_division=0.0
))

User-supplied MPS export; formatted from aggregate metrics, not executed in this session.
              precision    recall  f1-score   support

    negative       0.76      0.82      0.79        51
     neutral       0.93      0.84      0.88       297
    positive       0.73      0.86      0.79       137

Accuracy: 0.8412
Macro F1: 0.8199
Weighted F1: 0.8439


## Main rerun findings
The user-supplied MPS export records ten completed epochs for head-only adaptation and full fine-tuning on the original 485-example test split. Fine-tuned BERT reached 0.8412 accuracy, 0.8199 macro F1 and 0.8439 weighted F1, compared with historical rounded values 0.87, 0.84 and 0.87. The rerun improves accuracy by 8.04 percentage points over logistic regression. Validation loss selected epoch 1 for the full model (0.3956); later validation losses increased as training loss fell, indicating overfitting. Head-only adaptation selected epoch 10. Aggregate arithmetic, dataset hash, checkpoint revisions, baseline reports and logged checkpoint selection were checked. Raw predictions, model weights and the executed notebook were not supplied, so the training was not independently replayed here. Two unique test sentences also occur in training, and external FinBERT may have seen the evaluation corpus.


## External FinBERT: possible corpus overlap
The supplied MPS export reports 0.8907 accuracy, 0.8778 macro F1 and 0.8924 weighted F1. Financial PhraseBank may overlap the checkpoint training corpus, so these metrics are not independent evidence of generalization. Exact overlap was not audited.


In [ ]:
finbert = pipeline(model="ProsusAI/finbert", device=device, revision=finbert_revision)


y_pred_finbert = [pred["label"] for pred in finbert(X_test.to_list())]
print(classification_report(y_test, y_pred_finbert, zero_division=0.0))


User-supplied MPS export; formatted from aggregate metrics, not executed in this session.
              precision    recall  f1-score   support

    negative       0.80      0.94      0.86        51
     neutral       0.96      0.87      0.92       297
    positive       0.80      0.91      0.85       137

Accuracy: 0.8907
Macro F1: 0.8778
Weighted F1: 0.8924


## Reproducibility and limitations
- User-supplied MPS metrics and training logs were checked for aggregate consistency; the full training was not independently replayed here.
- Both training runs logged ten epochs. Best validation loss selected full-model epoch 1 and frozen-model epoch 10. Later full-model epochs showed overfitting.
- The split is small and non-stratified. Two unique test sentences also occur in training; no temporal holdout or confidence interval was evaluated.
- Fixed checkpoint revisions and seed 42 are recorded. Device and package differences were not isolated experimentally.
- External FinBERT may have prior exposure to evaluation sentences.
- Raw predictions and model weights are not bundled.

## Author
Cameron Batts


In [ ]:
# Run after all experiments finish. Exports aggregate metrics, not source sentences.
import json, hashlib, platform
from importlib.metadata import version
model_reports = {
    'Naive Bayes': classification_report(y_test, y_pred_nb, output_dict=True, zero_division=0),
    'Logistic regression': classification_report(y_test, y_pred_lr, output_dict=True, zero_division=0),
    'Off-the-shelf BERT': classification_report(y_test, y_pred_bert, output_dict=True, zero_division=0),
    'Frozen BERT': classification_report(y_test_tensor.numpy(), y_pred_frozen, labels=[0,1,2], target_names=['negative','neutral','positive'], output_dict=True, zero_division=0),
    'Fine-tuned BERT': classification_report(y_test_tensor.numpy(), y_pred_unfrozen, labels=[0,1,2], target_names=['negative','neutral','positive'], output_dict=True, zero_division=0),
    'External FinBERT (possible training overlap)': classification_report(y_test, y_pred_finbert, output_dict=True, zero_division=0),
}
record = {
    'python': platform.python_version(), 'device': device,
    'versions': {p: version(p) for p in ['numpy','pandas','scikit-learn','torch','transformers','accelerate']},
    'seed': 42, 'split_seed': 41, 'bert_revision': bert_revision, 'finbert_revision': finbert_revision,
    'dataset_sha256': hashlib.sha256((root/'data/all-data.csv').read_bytes()).hexdigest(),
    'test_rows': len(y_test), 'models': model_reports,
    'best_validation_checkpoints': {'frozen': trainer_frozen.state.best_model_checkpoint, 'full': trainer_unfrozen.state.best_model_checkpoint},
    'training_logs': {'frozen': trainer_frozen.state.log_history, 'full': trainer_unfrozen.state.log_history},
    'limitations': 'Single original split; duplicate texts retained; external FinBERT may have prior exposure to Financial PhraseBank.',
}
destination = root/'reports/gpu_rerun_metrics.json'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(record, indent=2) + '\n')
print('Saved aggregate metrics to:', destination)
